# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
    !pip install -q transformers torch pandas seaborn matplotlib
except ImportError:
    IN_COLAB = False

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Detecting AI-Generated Text In-Depth

As Large Language Models (LLMs) become ubiquitous, distinguishing between human and machine-generated text is a critical challenge in social science research.

Common hypotheses:
1.  **Perplexity (PPL)**: AI models aim to maximize probability, so they often choose "safe", predictable words. This leads to *lower* perplexity than human text.
2.  **Burstiness**: Humans write with varying sentence structures (short, then long and complex). AI tends to be more monotonic. Burstiness measures the variation in perplexity.

In this notebook, we will use a small model (**GPT-2**) to calculate these metrics on sample texts.


## 1. Loading the Evaluation Model

We use `gpt2` (small) as our reference model to calculate probability.


In [ ]:
model_id = "gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
model = GPT2LMHeadModel.from_pretrained(model_id).to(device)

print("Model loaded.")

## 2. Defining Metrics

We define a function to calculate Perplexity.
$$ PPL(X) = \exp \left( -\frac{1}{N} \sum_{i=1}^N \log P(x_i | x_{<i}) \right) $$


In [ ]:
def calculate_perplexity(text):
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss
        perplexity = torch.exp(loss)
    
    return perplexity.item()

def calculate_burstiness(text):
    # Split into sentences (simple split by period for demonstration)
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 5]
    if not sentences:
        return 0
    
    ppls = []
    for s in sentences:
        try:
            ppls.append(calculate_perplexity(s))
        except:
            pass
            
    if not ppls:
        return 0
        
    return np.std(ppls) # Burstiness is the standard deviation of sentence perplexities

print("Functions defined.")

## 3. Comparison: Human vs. AI

Let's test these metrics on two samples.


In [ ]:
# Sample 1: Human (Abstract from a paper, usually complex and varied)
human_text = """
The concept of social capital has generated considerable interest in recent decades. 
However, despite its popularity, the term remains elusive and is often defined in contradictory ways. 
In this paper, we argue that social capital should be understood primarily as a resource embedded in social networks. 
We demonstrate that this network-based perspective allows for more precise measurement and prediction of social outcomes.
"""

# Sample 2: AI (Generated by a simple model, usually very predictable)
ai_text = """
Social capital is a very important concept in sociology. It is used to describe the value of social networks. 
People can use social capital to get better jobs. It is also important for communities. 
When people trust each other, they have more social capital. This helps society function better.
"""

data = []

# Human
ppl_h = calculate_perplexity(human_text)
burst_h = calculate_burstiness(human_text)
data.append({'Type': 'Human', 'Perplexity': ppl_h, 'Burstiness': burst_h})

# AI
ppl_a = calculate_perplexity(ai_text)
burst_a = calculate_burstiness(ai_text)
data.append({'Type': 'AI', 'Perplexity': ppl_a, 'Burstiness': burst_a})

df = pd.DataFrame(data)
print(df)

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=df, x='Type', y='Perplexity', ax=ax[0], palette='pastel')
ax[0].set_title("Perplexity (Lower = More Predictable)")

sns.barplot(data=df, x='Type', y='Burstiness', ax=ax[1], palette='pastel')
ax[1].set_title("Burstiness (Lower = More Monotonic)")
plt.show()


## Conclusion

Typically, you will see that:
*   **AI Text** has lower Perplexity (it chooses probable words).
*   **AI Text** has lower Burstiness (it has a consistent style).
*   **Human Text** is "messier" (higher PPL) and more varied (higher Burstiness).

*Note: As AI models get better (like GPT-4), this gap narrows, making detection harder.*
